# Beauty-Axis Pseudo-Label Generation: Step-by-Step Visualization

Visualizes the complete pipeline from `src/pseudo_labels.py` — **no logic changes**.

| Step | Description |
|------|-------------|
| ① | Sample faces per ethnicity subgroup + graph maps |
| ② | Population mean face per ethnicity |
| ③ | Top-30% highest-rated faces + graph maps |
| ④ | Beauty prototype |
| ⑤ | Beauty axis (full face + per-organ) |
| ⑥ | Test face + graph map |
| ⑦ | Per-organ graphs of the test face |
| ⑧+⑨ | Per-organ histogram rank + test face position |
| ⑩ | Final pseudo-scores per organ |
| ⑪ | Geometric projection onto beauty axis (PCA 2D) |

> **Run cells in order.** Figures saved to `figures/`.

## Cell 0 : Google Colab Setup

Mount Drive, copy source `.py` files, and configure all paths.

> **Edit `DRIVE_ROOT`** if your Drive folder name is different.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil

# ── Edit this to match your Drive folder ────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/Colab Notebooks/FaceRankNet8'

PROJECT_ROOT = '/content/FaceRankNet'
os.makedirs(PROJECT_ROOT, exist_ok=True)

# Copy all .py source files from Drive to Colab runtime
copied = []
for fname in os.listdir(DRIVE_ROOT):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(DRIVE_ROOT, fname),
                    os.path.join(PROJECT_ROOT, fname))
        copied.append(fname)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Copied {len(copied)} .py files: {copied}')

CACHE    = f'{DRIVE_ROOT}/cache'
CSV      = f'{DRIVE_ROOT}/train_labels.csv'
SAVE_DIR = f'{DRIVE_ROOT}/figures'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'DRIVE_ROOT : {DRIVE_ROOT}')
print(f'CACHE      : {CACHE}')
print(f'CSV        : {CSV}')
print(f'SAVE_DIR   : {SAVE_DIR}')


In [ ]:
import pickle, bisect
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

from organ_indices import ORGAN_INDICES
from pseudo_labels import (
    compute_universal_average_face,
    compute_beauty_prototype,
    compute_beauty_axis,
    project_organ_onto_axis,
)

ORGAN_COLORS = {
    'left_eye':  '#2166AC',
    'right_eye': '#4393C3',
    'nose':      '#4DAC26',
    'mouth':     '#D6604D',
    'jawline':   '#762A83',
}
ORGAN_LABELS = {
    'left_eye':  'Left Eye',
    'right_eye': 'Right Eye',
    'nose':      'Nose',
    'mouth':     'Mouth',
    'jawline':   'Jawline',
}
ETH_COLORS = {'Asian': '#2166AC', 'Caucasian': '#D6604D'}

plt.rcParams.update({
    'font.family'      : 'serif',
    'font.serif'       : ['Times New Roman', 'DejaVu Serif', 'Palatino'],
    'font.size'        : 9,
    'axes.labelsize'   : 9,
    'axes.titlesize'   : 9,
    'xtick.labelsize'  : 8,
    'ytick.labelsize'  : 8,
    'legend.fontsize'  : 8,
    'legend.frameon'   : True,
    'legend.framealpha': 0.9,
    'legend.edgecolor' : '0.8',
    'axes.linewidth'   : 0.7,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'xtick.major.width': 0.7,
    'ytick.major.width': 0.7,
    'lines.linewidth'  : 1.0,
    'patch.linewidth'  : 0.5,
    'grid.linewidth'   : 0.4,
    'grid.color'       : '0.88',
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
})

def get_eth(f): return 'Asian' if f[0].upper() == 'A' else 'Caucasian'
def get_gen(f): return 'Female' if f[1].upper() == 'F' else 'Male'
def fxy(c):     return c[:, 0], -c[:, 1]

print('Setup complete.')
print(f'  Organs   : {list(ORGAN_INDICES.keys())}')
print(f'  Save dir : {SAVE_DIR}')

In [ ]:
# CACHE, CSV, SAVE_DIR are defined in Cell 0 (Colab Setup)
with open(f'{CACHE}/train_landmarks.pkl', 'rb') as fh:
    coords_cache = pickle.load(fh)
with open(f'{CACHE}/pseudo_labels.pkl', 'rb') as fh:
    pseudo_labels_ref = pickle.load(fh)
avg_face_ref = np.load(f'{CACHE}/avg_face.npy')

df           = pd.read_csv(CSV)
train_fnames = df['Filename'].tolist()
hol_ratings  = dict(zip(df['Filename'], df['Rating'].astype(float)))
# Use Ethnicity column if present, else derive from filename prefix
if 'Ethnicity' in df.columns:
    eth_map = dict(zip(df['Filename'], df['Ethnicity']))
    def get_eth(f): return eth_map.get(f, 'Asian' if f[0].upper()=='A' else 'Caucasian')
if 'Gender' in df.columns:
    gen_map = dict(zip(df['Filename'], df['Gender']))
    def get_gen(f): return gen_map.get(f, 'Female' if f[1].upper()=='F' else 'Male')
valid_train  = [f for f in train_fnames if f in coords_cache]

pop_mean    = {}
beauty_proto = {}
for eth in ['Asian', 'Caucasian']:
    eth_fnames  = [f for f in valid_train if get_eth(f) == eth]
    eth_coords  = [coords_cache[f] for f in eth_fnames]
    eth_ratings = [hol_ratings[f] for f in eth_fnames]
    pop_mean[eth]     = compute_universal_average_face(eth_coords)
    beauty_proto[eth] = compute_beauty_prototype(eth_coords, eth_ratings, top_k_pct=0.30)

beauty_axis = {
    eth: compute_beauty_axis(pop_mean[eth], beauty_proto[eth])
    for eth in pop_mean
}
pop_mean_global = compute_universal_average_face(
    [coords_cache[f] for f in valid_train]
)

organ_projs = {o: [] for o in ORGAN_INDICES}
proj_fnames = []
for fname in valid_train:
    eth  = get_eth(fname)
    mu   = pop_mean[eth]
    axis = beauty_axis[eth]
    proj_fnames.append(fname)
    for organ, idxs in ORGAN_INDICES.items():
        p = project_organ_onto_axis(coords_cache[fname], mu, axis, idxs)
        organ_projs[organ].append(p)

organ_sorted = {o: sorted(v) for o, v in organ_projs.items()}

print(f'Loaded {len(coords_cache)} faces  |  valid_train: {len(valid_train)}')
for eth in ['Asian', 'Caucasian']:
    n     = sum(1 for f in valid_train if get_eth(f) == eth)
    n_top = max(1, int(n * 0.30))
    print(f'  {eth:<12}: {n} faces  |  top-30% = {n_top}')
print('Projections computed.')

In [ ]:
JAWLINE = [10,338,297,332,284,251,389,356,454,323,361,288,
           397,365,379,378,400,377,152,148,176,149,150,136,
           172,58,132,93,234,127,162,21,54,103,67,109]
L_EYE   = [33,7,163,144,145,153,154,155,133,173,157,158,159,160,161,246]
R_EYE   = [362,382,381,380,374,373,390,249,263,466,388,387,386,385,384,398]
L_BROW  = [46,53,52,65,55,70,63,105,66,107]
R_BROW  = [276,283,282,295,285,300,293,334,296,336]
NOSE_B  = [168,197,195,5,4,1]
M_OUT   = [61,146,91,181,84,17,314,405,321,375,291,409,270,269,267,0,37,39,40,185]
M_IN    = [78,95,88,178,87,14,317,402,318,324,308,415,310,311,312,13,82,81,80,191]


def _path(ax, c, idx, col, lw=0.8, alpha=0.5, close=False):
    v = [i for i in idx if 0 <= i < 468]
    if len(v) < 2:
        return
    x, y = fxy(c)
    px, py = x[v], y[v]
    if close:
        px = np.append(px, px[0])
        py = np.append(py, py[0])
    ax.plot(px, py, color=col, lw=lw, alpha=alpha, zorder=2)


def draw_face(ax, c, title='', hl=None, ms=3, alpha_bg=0.15):
    x, y = fxy(c)
    _path(ax, c, JAWLINE, '#BBBBBB', lw=1.0, alpha=0.65, close=True)
    _path(ax, c, L_EYE,  ORGAN_COLORS['left_eye'],  lw=0.9, alpha=0.60, close=True)
    _path(ax, c, R_EYE,  ORGAN_COLORS['right_eye'], lw=0.9, alpha=0.60, close=True)
    _path(ax, c, L_BROW, ORGAN_COLORS['left_eye'],  lw=0.7, alpha=0.40)
    _path(ax, c, R_BROW, ORGAN_COLORS['right_eye'], lw=0.7, alpha=0.40)
    _path(ax, c, NOSE_B, ORGAN_COLORS['nose'],  lw=0.8, alpha=0.50)
    _path(ax, c, M_OUT,  ORGAN_COLORS['mouth'], lw=0.9, alpha=0.60, close=True)
    _path(ax, c, M_IN,   ORGAN_COLORS['mouth'], lw=0.5, alpha=0.30, close=True)
    for organ, idxs in ORGAN_INDICES.items():
        v  = [i for i in idxs if 0 <= i < 468]
        a  = 1.0 if hl is None or hl == organ else alpha_bg
        sz = (ms * 2.2)**2 if hl == organ else ms**2
        ax.scatter(x[v], y[v], c=ORGAN_COLORS[organ], s=sz,
                   alpha=a, zorder=3, linewidths=0)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    if title:
        ax.set_title(title, fontsize=9, pad=3)


def draw_organ(ax, c, organ, title='', color=None, ms=5, padding=0.35):
    col  = color or ORGAN_COLORS[organ]
    idxs = [i for i in ORGAN_INDICES[organ] if 0 <= i < 468]
    x, y = fxy(c)
    px, py = x[idxs], y[idxs]
    pts = np.column_stack([px, py])
    if len(pts) > 2:
        D = cdist(pts, pts)
        np.fill_diagonal(D, np.inf)
        k   = min(3, len(pts) - 1)
        thr = np.percentile(D[D < np.inf], 22)
        for i in range(len(pts)):
            for j in np.argsort(D[i])[:k]:
                if D[i, j] <= thr * 1.5:
                    ax.plot([px[i], px[j]], [py[i], py[j]],
                            color=col, lw=1.2, alpha=0.4, zorder=2)
    ax.scatter(px, py, c=col, s=ms**2, alpha=0.95, zorder=3,
               edgecolors='white', linewidths=0.4)
    if len(px) > 0:
        xr = max(px.ptp(), 1e-6)
        yr = max(py.ptp(), 1e-6)
        m  = max(xr, yr) * padding
        ax.set_xlim(px.min() - m, px.max() + m)
        ax.set_ylim(py.min() - m, py.max() + m)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ttl = title or ORGAN_LABELS.get(organ, organ)
    ax.set_title(ttl, fontsize=9, pad=3, color=col, fontweight='bold')


print('Helpers ready.')

## Step ① : Sample Faces per Ethnicity Subgroup

Shows 6 sample faces spanning the rating range (low → high) for each ethnicity.

In [ ]:
np.random.seed(42)
ETH_LIST = ['Asian', 'Caucasian']

fig, axes = plt.subplots(2, 6, figsize=(16, 6.5))
fig.suptitle(
    'Step ①  |  Sample Faces per Ethnicity Subgroup\n'
    '(MediaPipe FaceMesh 468 landmarks — color-coded by organ)',
    fontsize=9, fontweight='bold', y=1.02)

for row, eth in enumerate(ETH_LIST):
    eth_fnames = sorted([f for f in valid_train if get_eth(f) == eth],
                        key=lambda f: hol_ratings[f])
    n = len(eth_fnames)
    picks = [eth_fnames[int(n * q)] for q in [0.05, 0.20, 0.40, 0.60, 0.80, 0.95]]
    for col, fname in enumerate(picks):
        ax = axes[row, col]
        draw_face(ax, coords_cache[fname], ms=2.5)
        ax.set_title(f'Rating: {hol_ratings[fname]:.2f}', fontsize=8, pad=2,
                     color=ETH_COLORS[eth])
        if col == 0:
            ax.set_ylabel(eth, fontsize=11, fontweight='bold',
                          color=ETH_COLORS[eth], labelpad=4)
        ax.set_facecolor('#F5F9FF' if eth == 'Asian' else '#FFF8F0')

patches = [mpatches.Patch(facecolor=ORGAN_COLORS[o], label=ORGAN_LABELS[o])
           for o in ORGAN_INDICES]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.04), frameon=True, framealpha=0.9)
plt.tight_layout(h_pad=2.0, w_pad=0.4)
plt.savefig(f'{SAVE_DIR}/step1a_ethnicity_samples.pdf')
plt.savefig(f'{SAVE_DIR}/step1a_ethnicity_samples.png')
plt.show()
print('Saved: step1a_ethnicity_samples.pdf + .png')

In [ ]:
# Graph maps (transparent background, for overlay on actual face photos)
fig, axes = plt.subplots(2, 3, figsize=(9, 7))
fig.patch.set_alpha(0.0)
fig.suptitle('Step ①b  |  Graph Maps (transparent — overlay on real photos)',
             fontsize=11, fontweight='bold')

for row, eth in enumerate(ETH_LIST):
    eth_fnames = sorted([f for f in valid_train if get_eth(f) == eth],
                        key=lambda f: hol_ratings[f])
    n = len(eth_fnames)
    picks = [eth_fnames[int(n * 0.20)], eth_fnames[n // 2], eth_fnames[int(n * 0.85)]]
    for col, fname in enumerate(picks):
        ax = axes[row, col]
        ax.set_facecolor('none')
        x, y = fxy(coords_cache[fname])
        for organ, idxs in ORGAN_INDICES.items():
            v = [i for i in idxs if 0 <= i < 468]
            ax.scatter(x[v], y[v], c=ORGAN_COLORS[organ], s=10,
                       alpha=0.95, zorder=3, linewidths=0)
        _path(ax, coords_cache[fname], JAWLINE, '#777777', lw=1.0, alpha=0.55, close=True)
        _path(ax, coords_cache[fname], L_EYE,  ORGAN_COLORS['left_eye'],  lw=1.0, alpha=0.75, close=True)
        _path(ax, coords_cache[fname], R_EYE,  ORGAN_COLORS['right_eye'], lw=1.0, alpha=0.75, close=True)
        _path(ax, coords_cache[fname], M_OUT,  ORGAN_COLORS['mouth'],     lw=1.0, alpha=0.75, close=True)
        _path(ax, coords_cache[fname], NOSE_B, ORGAN_COLORS['nose'],      lw=0.9, alpha=0.65)
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.set_title(f'{eth}  •  Rating {hol_ratings[fname]:.2f}',
                     fontsize=8, pad=2, color=ETH_COLORS[eth])

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/step1b_graph_maps.png', transparent=True)
plt.show()
print('Saved: step1b_graph_maps.png (transparent)')

## Step ② : Population Mean per Ethnicity

`population_mean_e = mean(all train coords | ethnicity = e)` — the centroid of face-space.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5.5))
fig.suptitle(
    'Step ②  |  Population Mean Face per Ethnicity\n'
    'population_mean_e = mean( all train coords | ethnicity = e )',
    fontsize=9, fontweight='bold', y=1.02)

draw_face(axes[0], pop_mean_global, ms=3)
axes[0].set_title('Global Mean\n(all ethnicities)', fontsize=9, fontweight='bold', pad=5)
axes[0].set_facecolor('#F0F4F8')

for col, eth in enumerate(['Asian', 'Caucasian'], start=1):
    draw_face(axes[col], pop_mean[eth], ms=3)
    n = sum(1 for f in valid_train if get_eth(f) == eth)
    axes[col].set_title(f'{eth} Mean\n(n = {n} faces)',
                        fontsize=10, fontweight='bold', pad=5, color=ETH_COLORS[eth])
    axes[col].set_facecolor('#F8F9FA')

plt.tight_layout(w_pad=2.0)
plt.savefig(f'{SAVE_DIR}/step2_population_mean.pdf')
plt.savefig(f'{SAVE_DIR}/step2_population_mean.png')
plt.show()
print('Saved: step2_population_mean.pdf + .png')

## Step ③ : Top-30% Highest-Rated Faces per Ethnicity

These are the training faces whose holistic beauty rating places them in the **top 30%**. Their mean becomes the **beauty prototype**.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(16, 6.5))
fig.suptitle(
    'Step ③  |  Top-30% Highest-Rated Faces per Ethnicity\n'
    '(These faces define the beauty_prototype_e)',
    fontsize=9, fontweight='bold', y=1.02)

for row, eth in enumerate(ETH_LIST):
    eth_fnames  = [f for f in valid_train if get_eth(f) == eth]
    eth_ratings = [hol_ratings[f] for f in eth_fnames]
    n_top = max(1, int(len(eth_fnames) * 0.30))
    top_idx    = np.argsort(eth_ratings)[::-1]
    top_fnames = [eth_fnames[i] for i in top_idx[:6]]
    for col, fname in enumerate(top_fnames):
        ax = axes[row, col]
        draw_face(ax, coords_cache[fname], ms=2.5)
        star = '★' if col == 0 else ''
        ax.set_title(f'{star}#{col+1}: {hol_ratings[fname]:.2f}',
                     fontsize=8, pad=2, color=ETH_COLORS[eth])
        if col == 0:
            ax.set_ylabel(f'{eth}\n(top-{n_top})', fontsize=9,
                          fontweight='bold', color=ETH_COLORS[eth])
        ax.set_facecolor('#FFFEF0')

patches = [mpatches.Patch(facecolor=ORGAN_COLORS[o], label=ORGAN_LABELS[o])
           for o in ORGAN_INDICES]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.04))
plt.tight_layout(h_pad=2.0, w_pad=0.3)
plt.savefig(f'{SAVE_DIR}/step3_top_faces.pdf')
plt.savefig(f'{SAVE_DIR}/step3_top_faces.png')
plt.show()
print('Saved: step3_top_faces.pdf + .png')

## Step ④ : Beauty Prototype

`beauty_prototype_e = mean(top-30% rated face coords | ethnicity = e)`

Right panel: overlay of population mean (faint circles) vs beauty prototype (bright stars) for Asian.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5.5))
fig.suptitle(
    'Step ④  |  Beauty Prototype per Ethnicity\n'
    'beauty_prototype_e = mean( top-30% rated face coords | ethnicity = e )',
    fontsize=9, fontweight='bold', y=1.02)

for col, eth in enumerate(['Asian', 'Caucasian']):
    draw_face(axes[col], beauty_proto[eth], ms=3)
    n_top = max(1, int(sum(1 for f in valid_train if get_eth(f) == eth) * 0.30))
    axes[col].set_title(f'{eth} Beauty Prototype\n(mean of top-{n_top} faces)',
                        fontsize=9, fontweight='bold', color=ETH_COLORS[eth], pad=5)
    axes[col].set_facecolor('#FFFBF0')

# Overlay comparison (Asian)
ax  = axes[2]
eth = 'Asian'
xm, ym = fxy(pop_mean[eth])
xp, yp = fxy(beauty_proto[eth])
for organ, idxs in ORGAN_INDICES.items():
    v = [i for i in idxs if 0 <= i < 468]
    c = ORGAN_COLORS[organ]
    ax.scatter(xm[v], ym[v], color=c, s=7,  alpha=0.28, zorder=2, marker='o')
    ax.scatter(xp[v], yp[v], color=c, s=25, alpha=0.95, zorder=4, marker='*')
_path(ax, pop_mean[eth],    JAWLINE, '#CCCCCC', lw=1.0, alpha=0.40, close=True)
_path(ax, beauty_proto[eth], JAWLINE, ETH_COLORS[eth], lw=1.2, alpha=0.65, close=True)
ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
for sp in ax.spines.values(): sp.set_visible(False)
from matplotlib.lines import Line2D
leg = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='gray', ms=6,
           alpha=0.4, label='Population Mean'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='gold', ms=10,
           label='Beauty Prototype'),
]
ax.legend(handles=leg, fontsize=8, loc='lower center', framealpha=0.9)
ax.set_title('Asian: Mean ◦ vs Prototype ★\n(overlay)', fontsize=9,
             fontweight='bold', pad=5)
ax.set_facecolor('#F8F4FF')

plt.tight_layout(w_pad=2.0)
plt.savefig(f'{SAVE_DIR}/step4_beauty_prototype.pdf')
plt.savefig(f'{SAVE_DIR}/step4_beauty_prototype.png')
plt.show()
print('Saved: step4_beauty_prototype.pdf + .png')

## Step ⑤ : Beauty Axis

`beauty_axis_e = beauty_prototype_e − population_mean_e`

A (468×3) vector field pointing from the average face toward the attractive end of face-space. Arrows show the per-landmark shift direction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6.5))
fig.suptitle(
    'Step ⑤a  |  Beauty Axis: beauty_prototype_e − population_mean_e\n'
    'Arrows = direction each landmark shifts toward the attractive pole',
    fontsize=9, fontweight='bold', y=1.02)

for col, eth in enumerate(ETH_LIST):
    ax    = axes[col]
    mu    = pop_mean[eth]
    proto = beauty_proto[eth]
    axis  = beauty_axis[eth]
    xm, ym = fxy(mu)
    xp, yp = fxy(proto)
    xa = axis[:, 0]; ya = -axis[:, 1]

    # Population mean (faint)
    for organ, idxs in ORGAN_INDICES.items():
        v = [i for i in idxs if 0 <= i < 468]
        ax.scatter(xm[v], ym[v], c=ORGAN_COLORS[organ], s=4,
                   alpha=0.25, zorder=2, linewidths=0)
    _path(ax, mu, JAWLINE, '#CCCCCC', lw=1.0, alpha=0.45, close=True)

    # Arrows
    mags  = np.sqrt(xa**2 + ya**2)
    scale = 0.35 / (mags.max() + 1e-9)
    for i in range(0, 468, 12):
        if mags[i] > mags.mean() * 0.6:
            ax.annotate('',
                        xy=(xm[i] + xa[i]*scale, ym[i] + ya[i]*scale),
                        xytext=(xm[i], ym[i]),
                        arrowprops=dict(arrowstyle='->', color='#FF6B35',
                                        lw=0.9, alpha=0.65))

    # Beauty prototype (bright)
    for organ, idxs in ORGAN_INDICES.items():
        v = [i for i in idxs if 0 <= i < 468]
        ax.scatter(xp[v], yp[v], c=ORGAN_COLORS[organ], s=14,
                   alpha=0.90, zorder=4, marker='*', linewidths=0)
    _path(ax, proto, JAWLINE, ETH_COLORS[eth], lw=1.3, alpha=0.70, close=True)

    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_title(
        f'{eth} Beauty Axis\n'
        'faint = mean  |  bright ★ = prototype  |  → = beauty axis direction',
        fontsize=9, fontweight='bold', color=ETH_COLORS[eth], pad=5)
    ax.set_facecolor('#F8F9FA')
    from matplotlib.lines import Line2D
    leg = [
        Line2D([0],[0], marker='o', color='w', markerfacecolor='gray',
               ms=6, alpha=0.4, label='Population Mean'),
        Line2D([0],[0], marker='*', color='w', markerfacecolor='gold',
               ms=10, label='Beauty Prototype'),
        mpatches.Patch(color='#FF6B35', alpha=0.7, label='Beauty Axis →'),
    ]
    ax.legend(handles=leg, fontsize=7.5, loc='lower center', framealpha=0.9)

plt.tight_layout(w_pad=2.5)
plt.savefig(f'{SAVE_DIR}/step5a_beauty_axis_full.pdf')
plt.savefig(f'{SAVE_DIR}/step5a_beauty_axis_full.png')
plt.show()
print('Saved: step5a_beauty_axis_full.pdf + .png')

In [ ]:
# Per-organ beauty axis (Asian example)
eth   = 'Asian'
mu_5b = pop_mean[eth]
proto_5b = beauty_proto[eth]
axis_5b  = beauty_axis[eth]
xm5, ym5 = fxy(mu_5b)
xp5, yp5 = fxy(proto_5b)
xa5 = axis_5b[:, 0]; ya5 = -axis_5b[:, 1]
mags5 = np.sqrt(xa5**2 + ya5**2)

fig, axes = plt.subplots(1, 5, figsize=(16, 4.5))
fig.suptitle(
    'Step ⑤b  |  Per-Organ Beauty Axis  (ethnicity: Asian)\n'
    'beauty_axis_e[o] = organ-restricted subvector '
    'of (beauty_prototype_e − population_mean_e)',
    fontsize=9, fontweight='bold', y=1.02)

for col, organ in enumerate(ORGAN_INDICES.keys()):
    ax    = axes[col]
    idxs  = [i for i in ORGAN_INDICES[organ] if 0 <= i < 468]
    col_o = ORGAN_COLORS[organ]
    pxm, pym = xm5[idxs], ym5[idxs]
    pxp, pyp = xp5[idxs], yp5[idxs]
    pxa, pya = xa5[idxs], ya5[idxs]
    pmag = mags5[idxs]

    # Mean dots + connections
    pts = np.column_stack([pxm, pym])
    if len(pts) > 2:
        D = cdist(pts, pts); np.fill_diagonal(D, np.inf)
        thr = np.percentile(D[D<np.inf], 22)
        for i in range(len(pts)):
            for j in np.argsort(D[i])[:2]:
                if D[i,j] <= thr*1.5:
                    ax.plot([pxm[i],pxm[j]], [pym[i],pym[j]],
                            color='#AAAAAA', lw=0.8, alpha=0.4, zorder=1)
    ax.scatter(pxm, pym, c='#AAAAAA', s=28, alpha=0.55, zorder=2,
               edgecolors='white', linewidths=0.3, label='Mean')

    # Arrows
    sc = 0.25 / (pmag.max() + 1e-9)
    for i in range(len(idxs)):
        if pmag[i] > pmag.mean() * 0.4:
            ax.annotate('',
                        xy=(pxm[i]+pxa[i]*sc, pym[i]+pya[i]*sc),
                        xytext=(pxm[i], pym[i]),
                        arrowprops=dict(arrowstyle='->', color='#FF6B35',
                                        lw=0.9, alpha=0.75))

    # Prototype stars
    ax.scatter(pxp, pyp, c=col_o, s=60, alpha=0.92, zorder=4,
               edgecolors='white', linewidths=0.5, marker='*', label='Prototype')

    all_x = np.concatenate([pxm, pxp])
    all_y = np.concatenate([pym, pyp])
    xpad  = max(all_x.ptp(), 1e-6) * 0.55
    ypad  = max(all_y.ptp(), 1e-6) * 0.55
    ax.set_xlim(all_x.min()-xpad, all_x.max()+xpad)
    ax.set_ylim(all_y.min()-ypad, all_y.max()+ypad)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_title(f'{ORGAN_LABELS[organ]}\n({len(idxs)} nodes)',
                 fontsize=9, color=col_o, fontweight='bold', pad=3)
    if col == 0:
        ax.legend(fontsize=7, loc='upper left', framealpha=0.9, handlelength=1.2)

plt.tight_layout(w_pad=0.5)
plt.savefig(f'{SAVE_DIR}/step5b_beauty_axis_organs.pdf')
plt.savefig(f'{SAVE_DIR}/step5b_beauty_axis_organs.png')
plt.show()
print('Saved: step5b_beauty_axis_organs.pdf + .png')

## Step ⑥ : Select a Test Face

One training face is selected to demonstrate the full per-organ projection pipeline.

In [ ]:
np.random.seed(7)
asian_sorted = sorted([f for f in valid_train if get_eth(f) == 'Asian'],
                      key=lambda f: hol_ratings[f])
n_a = len(asian_sorted)
demo_fname = asian_sorted[int(n_a * 0.72)]

demo_eth  = get_eth(demo_fname)
demo_mu   = pop_mean[demo_eth]
demo_axis = beauty_axis[demo_eth]

demo_projs  = {}
demo_scores = {}
for organ, idxs in ORGAN_INDICES.items():
    proj = project_organ_onto_axis(coords_cache[demo_fname], demo_mu, demo_axis, idxs)
    demo_projs[organ] = proj
    rank  = bisect.bisect_left(organ_sorted[organ], proj) / len(organ_projs[organ])
    demo_scores[organ] = float(np.clip(1.0 + 4.0 * rank, 1.0, 5.0))

print(f'Demo face   : {demo_fname}')
print(f'Ethnicity   : {demo_eth}')
print(f'GT Rating   : {hol_ratings[demo_fname]:.3f}')
print(f'Mean pseudo : {np.mean(list(demo_scores.values())):.3f}')
print()
print(f'{"Organ":<14} {"Projection":>12}  {"Rank":>8}  {"Score":>7}')
print('-' * 46)
for organ in ORGAN_INDICES:
    proj = demo_projs[organ]
    rank = bisect.bisect_left(organ_sorted[organ], proj) / len(organ_projs[organ])
    print(f'{ORGAN_LABELS[organ]:<14} {proj:>12.4f}  {rank:>7.2%}  {demo_scores[organ]:>7.2f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
fig.suptitle(
    f'Step ⑥  |  Test Face: {demo_fname}\n'
    f'GT Rating = {hol_ratings[demo_fname]:.3f}  |  Ethnicity: {demo_eth}',
    fontsize=9, fontweight='bold')

draw_face(axes[0], coords_cache[demo_fname], ms=3)
axes[0].set_title('Full Face Landmark Graph\n(color-coded by organ region)', fontsize=9, loc='left')
axes[0].set_facecolor('#F8F9FA')

ax = axes[1]
ax.set_facecolor('none')
x, y = fxy(coords_cache[demo_fname])
for organ, idxs in ORGAN_INDICES.items():
    v = [i for i in idxs if 0 <= i < 468]
    ax.scatter(x[v], y[v], c=ORGAN_COLORS[organ], s=12, alpha=0.95, zorder=3, linewidths=0)
_path(ax, coords_cache[demo_fname], JAWLINE, '#777777', lw=1.2, alpha=0.60, close=True)
_path(ax, coords_cache[demo_fname], L_EYE,  ORGAN_COLORS['left_eye'],  lw=1.1, alpha=0.75, close=True)
_path(ax, coords_cache[demo_fname], R_EYE,  ORGAN_COLORS['right_eye'], lw=1.1, alpha=0.75, close=True)
_path(ax, coords_cache[demo_fname], M_OUT,  ORGAN_COLORS['mouth'],     lw=1.1, alpha=0.75, close=True)
_path(ax, coords_cache[demo_fname], NOSE_B, ORGAN_COLORS['nose'],      lw=1.0, alpha=0.65)
ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_title('Graph Map (transparent — overlay-ready)', fontsize=9, loc='left')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/step6_test_face.pdf')
plt.savefig(f'{SAVE_DIR}/step6_test_face.png')
plt.savefig(f'{SAVE_DIR}/step6_test_face_graphmap.png', transparent=True)
plt.show()
print('Saved: step6_test_face.pdf + .png  +  step6_test_face_graphmap.png')

## Step ⑦ : Per-Organ Landmark Subgraphs

Each organ is an independent subgraph of the full 468-node face. In the model these are **fully-connected GAT subgraphs**; here we show k-NN edges for clarity.

In [ ]:
fig = plt.figure(figsize=(16, 5.0))
fig.suptitle(
    f'Step ⑦  |  Per-Organ Landmark Subgraphs  (→ {demo_fname})\n'
    'Each organ processed independently by an OrganGAT sub-network',
    fontsize=9, fontweight='bold', y=1.02)

gs = gridspec.GridSpec(1, 5, figure=fig, wspace=0.12)
c  = coords_cache[demo_fname]

for col, organ in enumerate(ORGAN_INDICES.keys()):
    ax      = fig.add_subplot(gs[0, col])
    n_nodes = len([i for i in ORGAN_INDICES[organ] if i < 468])
    draw_organ(ax, c, organ, ms=5.5,
               title=f'{ORGAN_LABELS[organ]}\n{n_nodes} nodes')
    ax.set_facecolor('#FAFAFA')
    ax.text(0.5, -0.12, f'Score: {demo_scores[organ]:.2f}',
            transform=ax.transAxes, fontsize=9, ha='center',
            color=ORGAN_COLORS[organ], fontweight='bold')

plt.savefig(f'{SAVE_DIR}/step7_organ_graphs.pdf')
plt.savefig(f'{SAVE_DIR}/step7_organ_graphs.png')
plt.show()
print('Saved: step7_organ_graphs.pdf + .png')

## Steps ⑧+⑨ : Per-Organ Projection Histograms + Test Face Rank

For each organ *o*, collect all training-face projections, then locate the test face using `bisect.bisect_left`.

`rank = bisect_left(sorted_projections, test_proj) / N`  
`pseudo_score = clip(1 + 4 × rank, 1, 5)`

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
fig.suptitle(
    'Steps ⑧+⑨  |  Per-Organ Projection Distribution (all training faces)\n'
    'Red line = test face position  |  shaded = rank  |  '
    'score = clip(1 + 4 × rank, 1, 5)',
    fontsize=9, fontweight='bold', y=1.02)

for col, organ in enumerate(ORGAN_INDICES.keys()):
    ax        = axes[col]
    all_p     = organ_projs[organ]
    test_p    = demo_projs[organ]
    rank_val  = bisect.bisect_left(organ_sorted[organ], test_p) / len(all_p)
    score_val = demo_scores[organ]
    col_o     = ORGAN_COLORS[organ]

    cnts, edges, patches_h = ax.hist(
        all_p, bins=40, color=col_o, alpha=0.80,
        edgecolor='white', lw=0.3, zorder=2)

    # Color bars below test face
    mid = (edges[1] - edges[0]) / 2
    for patch, left in zip(patches_h, edges[:-1]):
        if left + mid < test_p:
            patch.set_facecolor(col_o); patch.set_alpha(0.65)
        else:
            patch.set_facecolor('#DDDDDD'); patch.set_alpha(0.80)

    y_max = cnts.max() if cnts.max() > 0 else 1
    ax.axvline(test_p, color='#C62828', lw=2.5, zorder=5)

    # Percentile reference lines
    for pct, lbl in [(0.25, '25%'), (0.50, '50%'), (0.75, '75%')]:
        pv = organ_sorted[organ][int(pct * len(organ_sorted[organ]))]
        ax.axvline(pv, color='#AAAAAA', lw=0.8, ls='--', alpha=0.6)
        ax.text(pv, -y_max*0.10, lbl, ha='center', fontsize=6, color='#999999')

    ha_ann = 'left' if rank_val > 0.5 else 'right'
    span   = edges[-1] - edges[0]
    xoff   = test_p + span*0.02 if rank_val < 0.5 else test_p - span*0.02
    ax.annotate(
        f'rank = {rank_val:.1%}\nscore = {score_val:.2f}',
        xy=(test_p, y_max * 0.88),
        xytext=(xoff, y_max * 0.88),
        fontsize=8, color='#C62828', fontweight='bold', ha=ha_ann,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#C62828', alpha=0.88))

    ax.set_xlabel('Projection  s(f, o)', fontsize=8)
    ax.set_title(f'{ORGAN_LABELS[organ]}\n{len(all_p)} training faces',
                 fontsize=9, color=col_o, fontweight='bold', pad=3)
    ax.set_ylim(-y_max*0.15, y_max*1.22)
    for sp in ['top','right','left']:
        ax.spines[sp].set_visible(False)
    ax.yaxis.set_visible(False)

plt.tight_layout(w_pad=0.5)
plt.savefig(f'{SAVE_DIR}/step8_9_histograms.pdf')
plt.savefig(f'{SAVE_DIR}/step8_9_histograms.png')
plt.show()
print('Saved: step8_9_histograms.pdf + .png')

## Step ⑩ : Final Pseudo-Scores per Organ

`p_{f,o} = 1 + 4 × rank_{f,o}  ∈ [1, 5]`

This is the pseudo-label vector **p_f = [p_{f,LE}, p_{f,RE}, p_{f,N}, p_{f,M}, p_{f,J}]** used as per-organ beauty targets during training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle(
    f'Step ⑩  |  Final Organ Pseudo-Scores  (→ {demo_fname})\n'
    f'p(f,o) = 1 + 4 × rank  ∈ [1,5]  |  GT Rating = {hol_ratings[demo_fname]:.3f}',
    fontsize=9, fontweight='bold', y=1.02)

organs = list(ORGAN_INDICES.keys())
scores = [demo_scores[o] for o in organs]
labels = [ORGAN_LABELS[o] for o in organs]
colors = [ORGAN_COLORS[o] for o in organs]

ax_bar = axes[0]
bars = ax_bar.bar(range(len(organs)), scores, color=colors,
                  linewidth=0, width=0.55, zorder=3)
mean_sc = np.mean(scores)
ax_bar.axhline(mean_sc, color='black', ls='--', lw=0.9, zorder=4,
               label=f'Mean Pseudo = {mean_sc:.2f}')
ax_bar.axhline(hol_ratings[demo_fname], color='0.45', ls=':', lw=0.9, zorder=4,
               label=f'GT Rating = {hol_ratings[demo_fname]:.2f}')
for bar, sc in zip(bars, scores):
    ax_bar.text(bar.get_x() + bar.get_width()/2, sc + 0.05,
               f'{sc:.2f}', ha='center', va='bottom', fontsize=8)
ax_bar.set_xticks(range(len(organs)))
ax_bar.set_xticklabels(labels, rotation=15, ha='right', fontsize=8)
ax_bar.set_ylim(1, 5.5)
ax_bar.set_ylabel('Pseudo-Score  p(f, o)')
ax_bar.set_title(f'Organ Pseudo-Label Vector for {demo_fname}', loc='left')
ax_bar.legend(loc='upper left', handlelength=1.8)
ax_bar.yaxis.grid(True, zorder=0)
ax_bar.set_axisbelow(True)

# Summary table
ax_tab = axes[1]
ax_tab.axis('off')
td = []
for organ in organs:
    proj  = demo_projs[organ]
    rank  = bisect.bisect_left(organ_sorted[organ], proj) / len(organ_projs[organ])
    score = demo_scores[organ]
    td.append([ORGAN_LABELS[organ], f'{proj:.4f}', f'{rank:.1%}', f'{score:.2f}'])
td.append(['Mean', '', '', f'{mean_sc:.2f}'])

tbl = ax_tab.table(
    cellText=td,
    colLabels=['Organ', 'Projection', 'Rank', 'Score [1-5]'],
    loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
tbl.scale(1.2, 1.9)
for i, organ in enumerate(organs):
    for j in range(4):
        tbl[(i+1, j)].set_facecolor(ORGAN_COLORS[organ] + '22')
for j in range(4):
    tbl[(0, j)].set_facecolor('#E8EAF6')
    tbl[(0, j)].set_text_props(fontweight='bold')
ax_tab.set_title(
    f'Pseudo-Label Summary\n'
    f'p_f = {[f"{s:.2f}" for s in scores]}',
    fontsize=9, pad=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/step10_final_scores.pdf')
plt.savefig(f'{SAVE_DIR}/step10_final_scores.png')
plt.show()
print('Saved: step10_final_scores.pdf + .png')

## Step ⒪ : Geometric Projection onto Beauty Axis (PCA 2D)

For each organ *o*:

$$s_{f,o} = \frac{(\mathbf{x}_{f,o} - \boldsymbol{\mu}_{e,o}) \cdot \mathbf{v}_{e,o}}{\|\mathbf{v}_{e,o}\|}$$

PCA reduces the flattened organ coordinates to 2D so we can visualize: population distribution (colored dots), beauty axis arrow (orange), and the perpendicular projection of the test face (dashed red line).

In [ ]:
eth_pca  = demo_eth
mu_pca   = pop_mean[eth_pca]
proto_pca = beauty_proto[eth_pca]
axis_pca  = beauty_axis[eth_pca]

eth_fnames_pca  = [f for f in valid_train if get_eth(f) == eth_pca]
eth_ratings_pca = np.array([hol_ratings[f] for f in eth_fnames_pca])

fig, axes = plt.subplots(1, 5, figsize=(19, 5))
fig.suptitle(
    f'Step ⒪  |  Per-Organ Projection onto Beauty Axis  (ethnicity: {eth_pca})\n'
    '○ = pop. mean  |  ★ = beauty prototype  |  ◆ = test face  |  '
    '→ = beauty axis  |  color = holistic rating',
    fontsize=11, fontweight='bold', y=1.05)

for col, organ in enumerate(ORGAN_INDICES.keys()):
    ax   = axes[col]
    idxs = [i for i in ORGAN_INDICES[organ] if i < 468]

    organ_feats = np.array(
        [coords_cache[f][idxs].ravel() for f in eth_fnames_pca])
    pca    = PCA(n_components=2, random_state=42)
    pts_2d = pca.fit_transform(organ_feats)

    mu_2d    = pca.transform(mu_pca[idxs].ravel().reshape(1,-1))[0]
    proto_2d = pca.transform(proto_pca[idxs].ravel().reshape(1,-1))[0]
    demo_2d  = pca.transform(
        coords_cache[demo_fname][idxs].ravel().reshape(1,-1))[0]

    # Beauty axis in PCA space
    ax_feat_pt = (axis_pca[idxs].ravel() + mu_pca[idxs].ravel()).reshape(1,-1)
    axis_2d    = pca.transform(ax_feat_pt)[0] - mu_2d
    axis_norm  = np.linalg.norm(axis_2d)
    axis_unit  = axis_2d / axis_norm if axis_norm > 1e-9 else np.array([1.,0.])

    # Scatter: training faces colored by rating
    ax.scatter(pts_2d[:,0], pts_2d[:,1], c=eth_ratings_pca,
               cmap='RdYlGn', s=7, alpha=0.40,
               vmin=1, vmax=5, zorder=2, linewidths=0)

    span    = max(pts_2d[:,0].ptp(), pts_2d[:,1].ptp(), 1e-6)
    arr_len = span * 0.30
    ax.annotate('',
                xy=(mu_2d[0]+axis_unit[0]*arr_len,
                    mu_2d[1]+axis_unit[1]*arr_len),
                xytext=(mu_2d[0]-axis_unit[0]*arr_len*0.25,
                        mu_2d[1]-axis_unit[1]*arr_len*0.25),
                arrowprops=dict(arrowstyle='->', color='#FF6B35',
                                lw=2.5, alpha=0.9))
    ax.text(mu_2d[0]+axis_unit[0]*arr_len*1.20,
            mu_2d[1]+axis_unit[1]*arr_len*1.20,
            'Beauty\nAxis', fontsize=6.5, color='#FF6B35',
            fontweight='bold', ha='center')

    # Key markers
    ax.scatter(*mu_2d,    c='#222222', s=100, zorder=5, marker='o',
               edgecolors='white', linewidths=1.5)
    ax.scatter(*proto_2d, c='gold',    s=180, zorder=6, marker='*',
               edgecolors='#E65100', linewidths=1.5)
    ax.scatter(*demo_2d,  c='#C62828', s=130, zorder=7, marker='D',
               edgecolors='white', linewidths=1.5)

    # Perpendicular projection line
    offset   = demo_2d - mu_2d
    proj_sc  = np.dot(offset, axis_unit)
    proj_pt  = mu_2d + proj_sc * axis_unit
    ax.plot([demo_2d[0], proj_pt[0]], [demo_2d[1], proj_pt[1]],
            color='#C62828', ls='--', lw=1.3, alpha=0.80, zorder=4)
    ax.scatter(*proj_pt, c='#C62828', s=70, zorder=5,
               marker='|', linewidths=2.5, alpha=0.9)

    ev = pca.explained_variance_ratio_
    ax.set_xlabel(f'PC1 ({ev[0]:.0%})', fontsize=7)
    ax.set_ylabel(f'PC2 ({ev[1]:.0%})', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.set_title(
        f'{ORGAN_LABELS[organ]}\n'
        f'proj={demo_projs[organ]:.4f}  score={demo_scores[organ]:.2f}',
        fontsize=9, color=ORGAN_COLORS[organ], fontweight='bold', pad=3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if col == 0:
        from matplotlib.lines import Line2D
        leg = [
            Line2D([0],[0], marker='o', color='w', markerfacecolor='#222', ms=7,
                   label='Pop. Mean'),
            Line2D([0],[0], marker='*', color='w', markerfacecolor='gold', ms=10,
                   label='Beauty Proto.'),
            Line2D([0],[0], marker='D', color='w', markerfacecolor='#C62828', ms=7,
                   label='Test Face'),
            mpatches.Patch(color='#FF6B35', label='Beauty Axis →'),
        ]
        ax.legend(handles=leg, fontsize=6, loc='upper left',
                  framealpha=0.9, handlelength=1.2)

sm = ScalarMappable(cmap='RdYlGn', norm=Normalize(vmin=1, vmax=5))
sm.set_array([])
cbar = plt.colorbar(sm, ax=axes, shrink=0.75, pad=0.01,
                    label='Holistic Rating', aspect=25)
cbar.ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/step11_projection_pca.pdf')
plt.savefig(f'{SAVE_DIR}/step11_projection_pca.png', bbox_inches='tight')
plt.show()
print('Saved: step11_projection_pca.pdf + .png')

## Summary

In [ ]:
import glob
print('=' * 60)
print('  Beauty-Axis Pseudo-Label Visualization: Summary')
print('=' * 60)
print(f'  Demo face : {demo_fname}')
print(f'  Ethnicity : {demo_eth}   |   GT Rating : {hol_ratings[demo_fname]:.3f}')
print()
print(f'  {"Organ":<14} {"Projection":>12}  {"Percentile":>12}  {"Score":>7}')
print('  ' + '-' * 50)
for organ in ORGAN_INDICES:
    proj  = demo_projs[organ]
    rank  = bisect.bisect_left(organ_sorted[organ], proj) / len(organ_projs[organ])
    score = demo_scores[organ]
    print(f'  {ORGAN_LABELS[organ]:<14} {proj:>12.4f}  {rank:>11.2%}  {score:>7.2f}')
print('  ' + '-' * 50)
print(f'  {"Mean":<14} {"":>12}  {"":>12}  '
      f'{np.mean(list(demo_scores.values())):>7.2f}')
print()
print(f'Figures in: {os.path.abspath(SAVE_DIR)}/')
for fp in sorted(glob.glob(f'{SAVE_DIR}/*.png')):
    print(f'  {os.path.basename(fp)}')
print('\nAll done!')